# Assignment 1: Skin Disease Classification Using Deep Learning
**Course**: Fundamentals of Deep Learning (Week 4)  
**Program**: Master of Computer Science / Information Technology (S2), President University  
**Author**: Aristo  
**Topic**: Comparative Evaluation of Transfer Learning CNNs (VGG16, ResNet50, MobileNetV2) on Dermoscopic Skin Lesions  

---

## 1. Project Background & Research Flowchart

Skin diseases encompass a wide spectrum of conditions ranging from benign keratosis and fungal infections to life-threatening malignancies such as melanoma. Early and accurate diagnosis using computer-aided dermoscopy systems can significantly assist healthcare providers in triage and early intervention.

In this study, we design, implement, and rigorously evaluate **three distinct deep convolutional neural network (CNN) architectures**:
1. **VGG16**: Deep sequential architecture with stacked 3x3 convolutions.
2. **ResNet50**: Deep residual network utilizing identity shortcut connections to alleviate vanishing gradients.
3. **MobileNetV2**: Efficient lightweight architecture utilizing inverted residual blocks and depthwise separable convolutions.

Below is the complete research methodology flowchart illustrating every stage from data loading, preprocessing, transfer learning, to evaluation.

In [1]:
import os
from IPython.display import Image, display

# Display Methodology Flowchart
flowchart_path = 'results/figures/research_flowchart.png'
if os.path.exists(flowchart_path):
    display(Image(filename=flowchart_path))
else:
    print("Flowchart not found at", flowchart_path)

## 2. Dataset & Exploratory Data Analysis (EDA)

We utilize the **Skin Disease Classification Image Dataset** which comprises 9 distinct diagnostic categories:
1. Actinic keratosis (Solar keratosis - precancerous)
2. Atopic Dermatitis (Eczema)
3. Benign keratosis (Seborrheic keratosis)
4. Dermatofibroma (Benign fibrous histiocytoma)
5. Melanocytic nevus (Common benign mole)
6. Melanoma (Malignant skin cancer)
7. Squamous cell carcinoma (Invasive keratinocyte carcinoma)
8. Tinea Ringworm Candidiasis (Fungal infection)
9. Vascular lesion (Hemangioma / angioma)

We inspect class frequencies, image integrity, and spatial dimensions.

In [2]:
import pandas as pd

# Display EDA Class Distribution Summary
eda_summary_path = 'results/eda_summary.md'
if os.path.exists(eda_summary_path):
    with open(eda_summary_path, 'r', encoding='utf-8') as f:
        print(f.read())

# Display Distribution Plot & Sample Images
display(Image(filename='results/figures/eda_class_distribution.png'))
display(Image(filename='results/figures/eda_sample_images.png'))

## 3. Data Preprocessing & Augmentation Strategy

### Preprocessing Steps:
1. **Resizing**: All dermoscopic images are resized to uniform spatial dimensions of `224 x 224` pixels to fit the canonical input dimensions of ImageNet-pretrained CNN backbones.
2. **Normalization**: Channel-wise pixel values are converted to tensors $[0, 1]$ and normalized using ImageNet population statistics:
   $$\mu = [0.485, 0.456, 0.406], \quad \sigma = [0.229, 0.224, 0.225]$$
3. **Data Augmentation**: To prevent overfitting and enhance generalization on medical images with varying orientations, random rotations ($\pm 15^\circ$), horizontal flips ($p=0.5$), and mild color jitter (brightness and contrast $\pm 10\%$) are applied **strictly to the training set**.
4. **Stratified Splitting**: The dataset is split into:
   - **Train**: 696 images (79.5%)
   - **Validation**: 91 images (10.4%)
   - **Test**: 89 images (10.2%)
   Validation and test sets do not undergo augmentation to guarantee unbiased assessment.

In [3]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

dataset_dir = "skin_disease_classification/Split_smol"

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

train_dataset = datasets.ImageFolder(os.path.join(dataset_dir, "train"), transform=train_transform)
val_dataset = datasets.ImageFolder(os.path.join(dataset_dir, "val"), transform=eval_transform)
test_dataset = datasets.ImageFolder(os.path.join(dataset_dir, "test"), transform=eval_transform)

print(f"Classes ({len(train_dataset.classes)}): {train_dataset.classes}")
print(f"Samples -> Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

## 4. Transfer Learning CNN Architectures

We instantiate three different convolutional paradigms with pretrained ImageNet weights:
- **VGG16**: Classic 16-layer network with homogeneous $3\times 3$ filters. Frozen features, followed by dense FC adapter with Dropout(0.5).
- **ResNet50**: 50-layer network utilizing Bottleneck residual blocks with skip connections. Replaces final FC layer with Dense(256) -> Dropout(0.5) -> Dense(9).
- **MobileNetV2**: 53-layer mobile architecture with inverted residual blocks and linear bottlenecks. Replaces classifier with Dropout(0.2) -> Dense(256) -> Dropout(0.4) -> Dense(9).

In [4]:
import torch.nn as nn
from torchvision.models import vgg16, resnet50, mobilenet_v2

num_classes = len(train_dataset.classes)

# Verify model definitions
def build_vgg16(num_classes):
    m = vgg16(weights='DEFAULT')
    for p in m.features.parameters(): p.requires_grad = False
    m.classifier = nn.Sequential(nn.Linear(m.classifier[0].in_features, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, num_classes))
    return m

def build_resnet50(num_classes):
    m = resnet50(weights='DEFAULT')
    for p in m.parameters(): p.requires_grad = False
    m.fc = nn.Sequential(nn.Linear(m.fc.in_features, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, num_classes))
    return m

def build_mobilenetv2(num_classes):
    m = mobilenet_v2(weights='DEFAULT')
    for p in m.features.parameters(): p.requires_grad = False
    m.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(m.classifier[1].in_features, 256), nn.ReLU(), nn.Dropout(0.4), nn.Linear(256, num_classes))
    return m

print("Model builders successfully defined.")

## 5. Model Training & Evaluation Results

Each model was trained for 10 epochs using Adam optimizer ($lr=10^{-4}$) and Cross-Entropy loss. Best validation checkpoint weights were automatically saved and subsequently loaded for final evaluation on the independent **Test Set** (89 images).

In [5]:
# Display Individual Training Curves
display(Image(filename='results/figures/vgg16_training_curves.png'))
display(Image(filename='results/figures/resnet50_training_curves.png'))
display(Image(filename='results/figures/mobilenetv2_training_curves.png'))

# Display Combined Validation Comparison Curves
display(Image(filename='results/figures/combined_validation_curves.png'))

## 6. Confusion Matrix & Per-Class Diagnostic Performance

Confusion matrices reveal specific classification dynamics across the 9 dermoscopic classes.

In [6]:
# Display All Confusion Matrices
display(Image(filename='results/figures/all_confusion_matrices.png'))

## 7. Comparative Performance Table

Below is the comprehensive comparison table summarizing Test Accuracy, Macro Precision, Macro Recall, Macro F1-Score, Model Parameters, and Training Wall-Clock Time.

In [7]:
comparison_df = pd.read_csv('results/model_comparison_table.csv')
display(comparison_df)

## 8. In-Depth Discussion & Error Analysis

### 1. Performance vs. Architectural Complexity
- **ResNet50 & VGG16** achieved identical top test accuracy (**75.28%**), with ResNet50 leading slightly in macro F1-score (**0.7521** vs. 0.7450).
- However, ResNet50 is **2.1x faster to train** (601s vs. 1259s for VGG16) and contains only **526,857 trainable parameters** compared to **6,425,097** in VGG16. This is primarily attributed to ResNet's Global Average Pooling prior to the head, which eliminates the heavy parameter burst inherent to VGG's flattened feature vector.
- **MobileNetV2** demonstrates exceptional computational efficiency: training in only **218.4s** (5.8x faster than VGG16) with only **2.55M total parameters** (10x smaller than ResNet50). While test accuracy was 66.29%, its lightweight footprint makes it the premier candidate for edge and mobile dermoscopy applications.

### 2. Clinical Confusion Analysis
- **Squamous Cell Carcinoma vs. Actinic Keratosis**: All models exhibited cross-confusion between Squamous Cell Carcinoma and Actinic Keratosis. In clinical dermatology, actinic keratosis is biologically recognized as the precursor lesion to invasive cutaneous squamous cell carcinoma, presenting near-identical histological hyperkeratosis and erythematous macules.
- **Melanocytic Nevus vs. Melanoma**: Both lesion types share melanin pigment networks and border features. Transfer learning features helped classify the majority accurately (8/10 for Melanoma in VGG16, 7/10 in ResNet50), but fine-tuning intermediate convolutional layers could further resolve borderline atypical nevi.
- **Benign Keratosis & Vascular Lesions**: Models achieved 90%–100% precision and recall on these categories due to highly distinct visual cues (stuck-on verrucous appearance for keratosis, and characteristic red-purple lacunae for vascular lesions).

## 9. Conclusion & References

### Conclusion
1. Successfully developed and compared three deep CNN models (VGG16, ResNet50, MobileNetV2) for 9-class skin disease classification using transfer learning.
2. ResNet50 emerged as the most balanced and effective architecture, achieving **75.28% accuracy** and **0.7521 F1-score** with rapid convergence and parameter efficiency.
3. MobileNetV2 offers a 10x parameter reduction and 5.8x speedup, presenting strong feasibility for mobile and point-of-care clinical deployment.

### References
1. Simonyan, K., & Zisserman, A. (2014). Very deep convolutional networks for large-scale image recognition. *arXiv preprint arXiv:1409.1556*.
2. He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. In *CVPR* (pp. 770-778).
3. Sandler, M., Howard, A., Zhu, M., Zhmoginov, A., & Chen, L. C. (2018). MobileNetV2: Inverted residuals and linear bottlenecks. In *CVPR* (pp. 4510-4520).
4. Tschandl, P., Rosendahl, C., & Kittler, H. (2018). The HAM10000 dataset, a large collection of multi-source dermatoscopic images of common pigmented skin lesions. *Scientific Data*, 5(1), 1-9.

---  
**Source Code Repository**:  
[https://github.com/aristo/skin-disease-classification](https://github.com/aristo/skin-disease-classification)